In [1]:
from pathlib import Path
import json as _json
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

# ── CONFIGURE HERE ──────────────────────────────────────────────────────────
DATA_DIR = Path(r"C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned")
GRID_DIR = DATA_DIR / "_grid_outputs"

# Hardened CSVs (50 seeds, clip=70th pct) — used for MAE / CV / alarm / trend
HARDENED_FILES = {
    "cumulative": GRID_DIR / "cumulative" / "metrics_csv" / "results_cumulative_hardened.csv",
    "disjoint":   GRID_DIR / "disjoint"  / "metrics_csv" / "results_disjoint_hardened.csv",
    "sliding":    GRID_DIR / "sliding"   / "metrics_csv" / "results_sliding_hardened.csv",
}
# Multi-pct CSVs (7 seeds, 4 clip pcts) — used for clipping sensitivity analysis only
MULTI_FILES = {
    "cumulative": GRID_DIR / "cumulative" / "metrics_csv" / "results_cumulative_multi.csv",
    "disjoint":   GRID_DIR / "disjoint"  / "metrics_csv" / "results_disjoint_multi.csv",
    "sliding":    GRID_DIR / "sliding"   / "metrics_csv" / "results_sliding_multi.csv",
}

BOUNDED_DIR = DATA_DIR / "_grid_outputs" / "cumulative" / "bounded"
DP_PARQ_DIR = DATA_DIR / "_grid_outputs" / "cumulative" / "dp_outputs"
PLOTS_DIR   = DATA_DIR / "eval_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MECH_COLORS = {"naive": "#e15759", "tree": "#4e79a7", "smooth": "#59a14f"}
METRICS     = ["ALERT_COUNT", "TOTAL_FLOWS", "TOTAL_BYTES"]
EPS_TOTALS  = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]

# v2 columns that may be absent in old CSVs (graceful fallback)
V2_COLS = ["TREND_flip", "TREND_fp_rate", "TREND_fn_rate",
           "TOPK", "OVERLAP_at_k", "KENDALL_tau"]
# ────────────────────────────────────────────────────────────────────────────

## eval_cyber — Cross-Strategy Evaluation  (v3)

Two CSV sources:
- **Hardened** (`results_*_hardened.csv`): 50 seeds, clip=70th pct — used for all main analyses
- **Multi-pct** (`results_*_multi.csv`): 7 seeds, 4 clip pcts — used for clipping sensitivity only

Sections produced:

1. **Per-KPI MAE summary** — mechanism rankings by strategy  *(hardened)*
2. **ε vs MAE curves with ±1 std bands** — uncertainty across 50 seeds  *(hardened)*
3. **Seed CV analysis** — coefficient of variation; compare 7-seed vs 50-seed stability  *(both)*
4. **Post-hoc alarm analysis** — recomputed at q25 threshold on saved parquets  *(hardened)*
5. **T-scaling analysis** — empirical vs theoretical Naive/Smooth noise ratio  *(hardened)*
6. **Safe zone heatmaps** — ε × window_days, MAE < 10% of truth mean  *(hardened)*
7. **Trend analysis** — TREND_flip / fp_rate / fn_rate by mechanism & ε  *(hardened)*
8. **Ranking analysis** — OVERLAP@k and Kendall τ  *(hardened)*
9. **Clipping sensitivity** — TOTAL_BYTES MAE by clip percentile  *(multi-pct)*

In [2]:
def _load_csv_set(file_dict, label):
    '''Load a set of per-strategy CSVs and concat into one DataFrame.'''
    parts = {}
    for strat, path in file_dict.items():
        if not path.exists():
            print(f"  WARNING [{label}]: {path.name} not found")
            continue
        df = pd.read_csv(path)
        df["strategy"] = strat
        for col in V2_COLS:
            if col not in df.columns:
                df[col] = np.nan
        parts[strat] = df
        n_seeds  = df["seed"].nunique()
        has_v2   = df["TREND_flip"].notna().any()
        clip_pcts = sorted(df["bytes_clip_pct"].unique().tolist())
        print(f"  [{label}] {strat}: shape={df.shape}  seeds={n_seeds}"
              f"  clip_pcts={clip_pcts}  v2={'yes' if has_v2 else 'no'}")
    return pd.concat(parts.values(), ignore_index=True) if parts else pd.DataFrame()

print("Loading hardened CSVs (primary — 50 seeds, clip=70th pct):")
all_results = _load_csv_set(HARDENED_FILES, "hardened")

print("\nLoading multi-pct CSVs (clipping sensitivity — 7 seeds, 4 clip pcts):")
multi_results = _load_csv_set(MULTI_FILES, "multi")

# Fall back to multi if hardened not yet available
if all_results.empty:
    print("\nHardened CSVs not yet available — falling back to multi-pct for all analyses.")
    all_results = multi_results.copy()

print(f"\nPrimary dataset (hardened): {all_results.shape}  |  {all_results['seed'].nunique()} seeds")
if not multi_results.empty:
    print(f"Multi-pct dataset:          {multi_results.shape}  |  {multi_results['seed'].nunique()} seeds")

Loading hardened CSVs (primary — 50 seeds, clip=70th pct):
  WARNING [hardened]: results_cumulative_hardened.csv not found
  WARNING [hardened]: results_disjoint_hardened.csv not found
  WARNING [hardened]: results_sliding_hardened.csv not found

Loading multi-pct CSVs (clipping sensitivity — 7 seeds, 4 clip pcts):
  [multi] cumulative: shape=(50400, 28)  seeds=50  clip_pcts=[70, 80, 90, 95]  v2=yes
  [multi] disjoint: shape=(113400, 28)  seeds=50  clip_pcts=[70, 80, 90, 95]  v2=yes
  [multi] sliding: shape=(163800, 28)  seeds=50  clip_pcts=[70, 80, 90, 95]  v2=yes

Hardened CSVs not yet available — falling back to multi-pct for all analyses.

Primary dataset (hardened): (327600, 28)  |  50 seeds
Multi-pct dataset:          (327600, 28)  |  50 seeds


## 1 — Per-KPI MAE Summary by Mechanism

In [3]:
mae_summary = (
    all_results
    .groupby(["strategy", "metric", "mechanism"])["MAE"]
    .agg(mean_MAE="mean", std_MAE="std", n="count")
    .reset_index()
    .sort_values(["strategy", "metric", "mean_MAE"])
)
print(mae_summary.to_string(index=False))

  strategy      metric mechanism     mean_MAE      std_MAE     n
cumulative ALERT_COUNT    smooth     1.983169     4.254222  5600
cumulative ALERT_COUNT      tree     2.272773     4.237515  5600
cumulative ALERT_COUNT     naive     2.538654     4.769808  5600
cumulative TOTAL_BYTES    smooth 14655.986978 39776.593162  5600
cumulative TOTAL_BYTES      tree 17703.409219 41357.432411  5600
cumulative TOTAL_BYTES     naive 20230.778926 46568.409441  5600
cumulative TOTAL_FLOWS    smooth     2.286731     4.509155  5600
cumulative TOTAL_FLOWS      tree     2.653476     4.559683  5600
cumulative TOTAL_FLOWS     naive     3.164056     5.421898  5600
  disjoint ALERT_COUNT    smooth     1.470090     3.235287 12600
  disjoint ALERT_COUNT      tree     1.802184     3.566636 12600
  disjoint ALERT_COUNT     naive     1.920514     3.828855 12600
  disjoint TOTAL_BYTES    smooth 11091.260553 30815.682276 12600
  disjoint TOTAL_BYTES      tree 13869.736490 34428.846310 12600
  disjoint TOTAL_BYTES   

## 2 — ε vs MAE Curves (±1 std across seeds)

In [4]:
eps_summary = (
    all_results
    .groupby(["metric", "mechanism", "eps_total"])["MAE"]
    .agg(mean_MAE="mean", std_MAE="std")
    .reset_index()
)

fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 4), sharey=False)

for ax, metric in zip(axes, METRICS):
    for mech, color in MECH_COLORS.items():
        sub = eps_summary[eps_summary["mechanism"] == mech].sort_values("eps_total")
        if sub.empty:
            continue
        mu, sigma = sub["mean_MAE"].values, sub["std_MAE"].fillna(0).values
        ax.plot(sub["eps_total"], mu, marker="o", label=mech.capitalize(), color=color, lw=2)
        ax.fill_between(sub["eps_total"], (mu - sigma).clip(0), mu + sigma,
                        alpha=0.18, color=color)
    ax.set_xscale("log")
    ax.set_title(metric)
    ax.set_xlabel("ε (log scale)")
    ax.set_ylabel("MAE (mean ± 1 std)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("CyberSec — ε vs MAE by KPI (all strategies combined)", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eps_vs_mae.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {PLOTS_DIR / 'eps_vs_mae.png'}")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\eps_vs_mae.png


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\1784328889.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 — Seed Variance (Coefficient of Variation)

In [5]:
cv_df = (
    all_results
    .groupby(["metric", "mechanism", "eps_total"])["MAE"]
    .agg(mu="mean", sigma="std")
    .assign(cv=lambda d: d["sigma"] / d["mu"].replace(0, np.nan))
    .reset_index()
)

fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 4), sharey=False)
for ax, metric in zip(axes, METRICS):
    sub = cv_df[cv_df["metric"] == metric]
    for mech, color in MECH_COLORS.items():
        m = sub[sub["mechanism"] == mech].sort_values("eps_total")
        if m.empty:
            continue
        ax.plot(m["eps_total"], m["cv"], marker="s", label=mech.capitalize(),
                color=color, lw=2)
    ax.set_xscale("log")
    ax.set_title(metric)
    ax.set_xlabel("ε (log scale)")
    ax.set_ylabel("CV = std / mean")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("CyberSec — Seed Coefficient of Variation by KPI", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "seed_cv.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {PLOTS_DIR / 'seed_cv.png'}")

# print worst-CV rows
print("\nTop-10 highest-CV rows:")
print(cv_df.nlargest(10, "cv")[["metric","mechanism","eps_total","cv"]].to_string(index=False))

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\seed_cv.png

Top-10 highest-CV rows:
     metric mechanism  eps_total       cv
TOTAL_BYTES    smooth      16.00 1.678926
TOTAL_BYTES    smooth       2.00 1.645483
TOTAL_BYTES    smooth       0.50 1.641519
TOTAL_BYTES    smooth       1.00 1.618106
TOTAL_BYTES    smooth       0.25 1.592445
TOTAL_BYTES    smooth       8.00 1.574514
TOTAL_BYTES    smooth       4.00 1.568784
TOTAL_BYTES      tree      16.00 1.402172
TOTAL_BYTES      tree       1.00 1.391972
TOTAL_BYTES     naive       0.50 1.389828


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\1523684092.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 — Post-hoc Alarm Analysis

Re-computes alarm statistics on seed-0 saved parquets using the **q25 threshold**
(below-median days should raise alarms — conservative for a security KPI).

FN is penalised 5× (missed attack days cost more than false positives).

In [6]:
def alarm_stats_custom(truth_s, dp_s, threshold):
    '''Alarm statistics using a fixed threshold on a single slice.'''
    truth  = (truth_s > threshold).astype(int).to_numpy()
    pred   = (dp_s   > threshold).astype(int).to_numpy()
    total  = len(truth)
    fp     = int(np.sum((truth == 0) & (pred == 1)))
    fn     = int(np.sum((truth == 1) & (pred == 0)))
    return {
        "flip":    (fp +     fn) / total if total else np.nan,
        "wflip":   (fp + 5 * fn) / total if total else np.nan,
        "fp_rate":  fp           / total if total else np.nan,
        "fn_rate":  fn           / total if total else np.nan,
    }

# Scan bounded parquets saved at seed_id == 0
bounded_files = list(BOUNDED_DIR.glob("bounded__*.parquet")) if BOUNDED_DIR.exists() else []
print(f"Found {len(bounded_files)} bounded truth parquets")

alarm_rows = []
for bp in bounded_files:
    meta_path = bp.with_suffix(".json")
    if not meta_path.exists():
        continue
    with open(meta_path) as fh:
        meta = _json.load(fh)

    truth_df = pd.read_parquet(bp)
    q25 = {m: float(truth_df[m].quantile(0.25)) for m in METRICS if m in truth_df.columns}

    for mech in ["naive", "tree", "smooth"]:
        for eps in EPS_TOTALS:
            tag  = bp.stem.replace("bounded__", "")
            dpf  = DP_PARQ_DIR / f"dp__{mech}__eps={eps}__{tag}.parquet"
            if not dpf.exists():
                continue
            dp_df = pd.read_parquet(dpf)

            colmap = {
                "ALERT_COUNT": "ALERT_COUNT_DP"     if mech == "naive" else (
                               "ALERT_COUNT_TREE_DP"   if mech == "tree" else "ALERT_COUNT_SMOOTH_DP"),
                "TOTAL_FLOWS": "TOTAL_FLOWS_DP"      if mech == "naive" else (
                               "TOTAL_FLOWS_TREE_DP"   if mech == "tree" else "TOTAL_FLOWS_SMOOTH_DP"),
                "TOTAL_BYTES": "TOTAL_BYTES_DP"      if mech == "naive" else (
                               "TOTAL_BYTES_TREE_DP"   if mech == "tree" else "TOTAL_BYTES_SMOOTH_DP"),
            }

            for metric in METRICS:
                if metric not in truth_df.columns or colmap[metric] not in dp_df.columns:
                    continue
                stats = alarm_stats_custom(truth_df[metric].astype(float),
                                           dp_df[colmap[metric]].astype(float),
                                           q25[metric])
                alarm_rows.append({
                    "mechanism": mech,
                    "eps_total": eps,
                    "metric":    metric,
                    **stats,
                })

if alarm_rows:
    alarm_df = pd.DataFrame(alarm_rows)
    alarm_summary = (
        alarm_df.groupby(["metric", "mechanism", "eps_total"])
        [["flip","wflip","fp_rate","fn_rate"]]
        .mean()
        .reset_index()
        .sort_values(["metric","mechanism","eps_total"])
    )
    print("\nAlarm stats summary (post-hoc q25 threshold, FN_WEIGHT=5):")
    print(alarm_summary.to_string(index=False))
else:
    print("No DP parquets found — run strategy notebooks first (seed_id=0 saves parquets).")
    alarm_summary = pd.DataFrame()

Found 20 bounded truth parquets

Alarm stats summary (post-hoc q25 threshold, FN_WEIGHT=5):
     metric mechanism  eps_total  flip  wflip  fp_rate  fn_rate
ALERT_COUNT     naive       0.25  0.17   0.17     0.17      0.0
ALERT_COUNT     naive       0.50  0.08   0.08     0.08      0.0
ALERT_COUNT     naive       1.00  0.30   0.30     0.30      0.0
ALERT_COUNT     naive       2.00  0.05   0.05     0.05      0.0
ALERT_COUNT     naive       4.00  0.05   0.05     0.05      0.0
ALERT_COUNT     naive       8.00  0.25   0.25     0.25      0.0
ALERT_COUNT     naive      16.00  0.30   0.30     0.30      0.0
ALERT_COUNT    smooth       0.25  0.17   0.17     0.17      0.0
ALERT_COUNT    smooth       0.50  0.09   0.09     0.09      0.0
ALERT_COUNT    smooth       1.00  0.30   0.30     0.30      0.0
ALERT_COUNT    smooth       2.00  0.05   0.05     0.05      0.0
ALERT_COUNT    smooth       4.00  0.25   0.25     0.25      0.0
ALERT_COUNT    smooth       8.00  0.00   0.00     0.00      0.0
ALERT_COUNT 

## 5 — T-Scaling Analysis (Naive vs Smooth Noise Ratio)

In [7]:
# Theoretical: Naive noise ∝ T/ε, Smooth Binary noise ∝ log2(T)/ε
# Predicted ratio = T / log2(T)

t_values = np.linspace(2, 400, 400)
theory_ratio = t_values / np.log2(t_values)

# Empirical points from the combined results
pivot = (
    all_results[all_results["mechanism"].isin(["naive", "smooth"])]
    .groupby(["mechanism", "window_days"])["MAE"]
    .mean()
    .unstack("mechanism")
    .dropna()
)
empirical_T   = pivot.index.values
empirical_ratio = (pivot["naive"] / pivot["smooth"].replace(0, np.nan)).values

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_values, theory_ratio, "--", color="gray", lw=2, label="Theory: T / log₂(T)")
ax.scatter(empirical_T, empirical_ratio, zorder=5, s=80, label="Empirical (CyberSec)",
           color="#4e79a7")

# EComm reference point T=365
ecomm_T = 365
ax.scatter([ecomm_T], [ecomm_T / math.log2(ecomm_T)], marker="*", s=180,
           color="orange", zorder=6, label=f"Theory EComm (T={ecomm_T})")

ax.set_xlabel("Window length T (days)")
ax.set_ylabel("Naive MAE / Smooth MAE")
ax.set_title("T-Scaling: Empirical vs Theoretical Naive/Smooth Ratio")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "t_scaling.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {PLOTS_DIR / 't_scaling.png'}")

print("\nEmpirical T-scaling table:")
for T, r in zip(empirical_T, empirical_ratio):
    theory = T / math.log2(T)
    print(f"  T={int(T):3d}  empirical={r:.2f}  theory={theory:.2f}  frac={r/theory:.2f}×theory")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\t_scaling.png

Empirical T-scaling table:
  T=  1  empirical=0.96  theory=inf  frac=0.00×theory
  T=  2  empirical=2.00  theory=2.00  frac=1.00×theory
  T=  3  empirical=1.28  theory=1.89  frac=0.68×theory
  T=  5  empirical=1.34  theory=2.15  frac=0.62×theory


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\2888036932.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\2888036932.py:40: RuntimeWarning: divide by zero encountered in divide
  theory = T / math.log2(T)


## 6 — Safe Zone Heatmaps (ε × window_days)

A cell is **safe** if mean MAE < 10% of the ground-truth daily mean for that KPI.
Blue rectangle outlines the safe operating zone.

In [8]:
SAFETY_THRESH_FRACTION = 0.10   # 10% of truth mean

# truth means per metric (use bounded truth at 90th percentile clip)
truth_means = {}
if BOUNDED_DIR.exists():
    for bp in BOUNDED_DIR.glob("bounded__*bytespct=90*.parquet"):
        df_t = pd.read_parquet(bp)
        for m in METRICS:
            if m in df_t.columns:
                truth_means.setdefault(m, []).extend(df_t[m].tolist())
truth_means = {m: np.mean(v) for m, v in truth_means.items()}
if not truth_means:
    # fall back to overall dataset means from sensitivity JSON
    sens_path = DATA_DIR / "cicids_sensitivity.json"
    if sens_path.exists():
        with open(sens_path) as fh:
            sens = _json.load(fh)
        truth_means = {
            "ALERT_COUNT": sens.get("total_attack_flows", 557646) / 5,
            "TOTAL_FLOWS":  sens.get("total_flows",       2830743) / 5,
            "TOTAL_BYTES":  sens.get("total_flows",       2830743) / 5 * 4000,
        }

heatmap_data = (
    all_results
    .groupby(["metric", "mechanism", "eps_total", "window_days"])["MAE"]
    .mean()
    .reset_index()
)

for mech in ["naive", "tree", "smooth"]:
    fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 4))
    for ax, metric in zip(axes, METRICS):
        sub = heatmap_data[
            (heatmap_data["mechanism"] == mech) &
            (heatmap_data["metric"]    == metric)
        ].copy()
        if sub.empty:
            ax.set_title(f"{metric}\n(no data)")
            continue

        pivot_h = sub.pivot(index="window_days", columns="eps_total", values="MAE")
        safe_thresh = truth_means.get(metric, np.nan) * SAFETY_THRESH_FRACTION

        im = ax.imshow(pivot_h.values, aspect="auto", cmap="YlOrRd_r", origin="lower")
        ax.set_xticks(range(len(pivot_h.columns)))
        ax.set_xticklabels([str(e) for e in pivot_h.columns], fontsize=8)
        ax.set_yticks(range(len(pivot_h.index)))
        ax.set_yticklabels([str(w) for w in pivot_h.index], fontsize=8)
        ax.set_xlabel("ε")
        ax.set_ylabel("window_days")
        ax.set_title(f"{metric}\n(safe < {safe_thresh:.0f})")
        plt.colorbar(im, ax=ax, label="MAE")

        # draw blue outlines for safe cells
        for ri, row_val in enumerate(pivot_h.index):
            for ci, col_val in enumerate(pivot_h.columns):
                cell_mae = pivot_h.loc[row_val, col_val]
                if not np.isnan(cell_mae) and cell_mae < safe_thresh:
                    rect = mpatches.Rectangle(
                        (ci - 0.5, ri - 0.5), 1, 1,
                        linewidth=2, edgecolor="blue", facecolor="none"
                    )
                    ax.add_patch(rect)

    plt.suptitle(f"Safe Zone Heatmap — {mech.capitalize()}", y=1.01, fontsize=12)
    plt.tight_layout()
    fname = PLOTS_DIR / f"safe_zone_{mech}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {fname}")

C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\1929818997.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\safe_zone_naive.png


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\1929818997.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\safe_zone_tree.png
Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\safe_zone_smooth.png


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\1929818997.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 — Trend Analysis  *(v2 columns)*

`trend_positive()` flags days where the KPI is increasing over a 2-day rolling window.
Compares trend predictions from DP output vs ground truth.

- **TREND_flip**: overall disagreement rate
- **TREND_fp_rate**: DP says "rising" when truth is not
- **TREND_fn_rate**: DP misses a real rising trend (higher cost for security)

In [9]:
if all_results["TREND_flip"].isna().all():
    print("TREND columns are NaN — re-run strategy notebooks with v2 config to populate these.")
else:
    trend_summary = (
        all_results
        .groupby(["metric", "mechanism", "eps_total"])
        [["TREND_flip", "TREND_fp_rate", "TREND_fn_rate"]]
        .mean()
        .reset_index()
        .sort_values(["metric", "mechanism", "eps_total"])
    )

    fig, axes = plt.subplots(1, len(METRICS), figsize=(5 * len(METRICS), 4))
    for ax, metric in zip(axes, METRICS):
        sub = trend_summary[trend_summary["metric"] == metric]
        for mech, color in MECH_COLORS.items():
            m = sub[sub["mechanism"] == mech].sort_values("eps_total")
            if m.empty:
                continue
            ax.plot(m["eps_total"], m["TREND_flip"],    marker="o",
                    color=color, lw=2, label=f"{mech.capitalize()} (flip)")
            ax.plot(m["eps_total"], m["TREND_fn_rate"], marker="^", ls="--",
                    color=color, lw=1.5, label=f"{mech.capitalize()} (fn_rate)", alpha=0.7)
        ax.set_xscale("log")
        ax.set_ylim(0, 1.05)
        ax.set_title(metric)
        ax.set_xlabel("ε (log scale)")
        ax.set_ylabel("Rate")
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.suptitle("CyberSec — Trend Flip / FN Rate by KPI", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "trend_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {PLOTS_DIR / 'trend_analysis.png'}")
    print("\nMechanism ranking by TREND_fn_rate (lower = better, per metric):")
    for metric in METRICS:
        sub = trend_summary[trend_summary["metric"] == metric]
        best = sub.loc[sub["TREND_fn_rate"].idxmin()] if not sub.empty else None
        if best is not None:
            print(f"  {metric}: best mech = {best['mechanism']}  "
                  f"(fn_rate={best['TREND_fn_rate']:.3f} at ε={best['eps_total']})")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\trend_analysis.png

Mechanism ranking by TREND_fn_rate (lower = better, per metric):
  ALERT_COUNT: best mech = naive  (fn_rate=0.000 at ε=0.25)
  TOTAL_FLOWS: best mech = naive  (fn_rate=0.000 at ε=0.25)
  TOTAL_BYTES: best mech = naive  (fn_rate=0.000 at ε=0.25)


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\3022063637.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8 — Ranking Analysis: OVERLAP@k and Kendall τ  *(v2 columns)*

- **OVERLAP@k**: fraction of the top-k days by true count that DP also identifies as top-k
- **Kendall τ**: rank correlation between DP-ranked days and truth-ranked days  (−1 to +1)

A higher OVERLAP@k and Kendall τ indicate that the DP mechanism correctly preserves
which days have the highest activity — critical for anomaly triage.

In [10]:
if all_results["OVERLAP_at_k"].isna().all():
    print("OVERLAP_at_k / KENDALL_tau are NaN — re-run strategy notebooks with v2 config.")
else:
    rank_summary = (
        all_results
        .groupby(["metric", "mechanism", "eps_total"])
        [["OVERLAP_at_k", "KENDALL_tau"]]
        .mean()
        .reset_index()
    )

    fig, axes = plt.subplots(2, len(METRICS),
                             figsize=(5 * len(METRICS), 8),
                             sharex="col")

    for col_idx, metric in enumerate(METRICS):
        sub = rank_summary[rank_summary["metric"] == metric]

        ax_ovk  = axes[0][col_idx]
        ax_ktau = axes[1][col_idx]

        for mech, color in MECH_COLORS.items():
            m = sub[sub["mechanism"] == mech].sort_values("eps_total")
            if m.empty:
                continue
            ax_ovk.plot(m["eps_total"],  m["OVERLAP_at_k"], marker="o",
                        color=color, lw=2, label=mech.capitalize())
            ax_ktau.plot(m["eps_total"], m["KENDALL_tau"],   marker="s",
                         color=color, lw=2)

        ax_ovk.set_xscale("log")
        ax_ovk.set_ylim(0, 1.05)
        ax_ovk.set_title(metric)
        ax_ovk.set_ylabel("OVERLAP@k")
        ax_ovk.legend(fontsize=8)
        ax_ovk.grid(True, alpha=0.3)

        ax_ktau.set_xscale("log")
        ax_ktau.set_ylim(-1, 1)
        ax_ktau.axhline(0, color="gray", lw=0.8, ls=":")
        ax_ktau.set_xlabel("ε (log scale)")
        ax_ktau.set_ylabel("Kendall τ")
        ax_ktau.grid(True, alpha=0.3)

    axes[0][0].set_ylabel("OVERLAP@k")
    axes[1][0].set_ylabel("Kendall τ")
    plt.suptitle("CyberSec — Ranking Preservation by KPI", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "ranking_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {PLOTS_DIR / 'ranking_analysis.png'}")

    # Print best mechanism per metric at eps_total=1.0 (typical operating point)
    print("\nRanking at ε=1.0 (typical operating point):")
    for metric in METRICS:
        sub1 = rank_summary[
            (rank_summary["metric"] == metric) &
            (rank_summary["eps_total"] == 1.0)
        ].set_index("mechanism")
        if sub1.empty:
            print(f"  {metric}: no data at ε=1.0")
            continue
        best_ovk  = sub1["OVERLAP_at_k"].idxmax()
        best_ktau = sub1["KENDALL_tau"].idxmax()
        print(f"  {metric}:  OVERLAP@k best={best_ovk} ({sub1.loc[best_ovk,'OVERLAP_at_k']:.3f})"
              f"  |  Kendall τ best={best_ktau} ({sub1.loc[best_ktau,'KENDALL_tau']:.3f})")

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\ranking_analysis.png

Ranking at ε=1.0 (typical operating point):
  ALERT_COUNT:  OVERLAP@k best=naive (1.000)  |  Kendall τ best=naive (1.000)
  TOTAL_FLOWS:  OVERLAP@k best=naive (1.000)  |  Kendall τ best=naive (1.000)
  TOTAL_BYTES:  OVERLAP@k best=naive (1.000)  |  Kendall τ best=naive (1.000)


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\1730136888.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Best ε per Metric & Mechanism (lowest mean MAE)

In [11]:
eps_group = (
    all_results
    .groupby(["metric", "mechanism", "eps_total"])["MAE"]
    .mean()
    .reset_index()
    .rename(columns={"MAE": "mean_MAE"})
)

best_eps_rows = []
for (metric, mech), grp in eps_group.groupby(["metric", "mechanism"]):
    idx_min = grp["mean_MAE"].idxmin()
    best_eps_rows.append(grp.loc[idx_min])

best_eps = pd.DataFrame(best_eps_rows).sort_values(["metric", "mean_MAE"])
print("Best ε per metric/mechanism:")
print(best_eps.to_string(index=False))

Best ε per metric/mechanism:
     metric mechanism  eps_total   mean_MAE
ALERT_COUNT    smooth       16.0   0.089883
ALERT_COUNT      tree       16.0   0.111747
ALERT_COUNT     naive       16.0   0.115522
TOTAL_BYTES    smooth       16.0 667.566068
TOTAL_BYTES      tree       16.0 840.981376
TOTAL_BYTES     naive       16.0 902.966095
TOTAL_FLOWS    smooth       16.0   0.099483
TOTAL_FLOWS      tree       16.0   0.121840
TOTAL_FLOWS     naive       16.0   0.134002


## Overall Mechanism Ranking — Composite Score

In [12]:
# Composite score: normalised MAE (lower = better) × 0.5
#                + (1 - OVERLAP@k) × 0.25
#                + TREND_fn_rate × 0.25
# (weights can be adjusted; OVERLAP and TREND fallback to 0 if v2 columns absent)

def safe_norm(s):
    '''Min-max normalise a series; returns 0.5 everywhere if constant.'''
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 1e-12 else pd.Series(0.5, index=s.index)

composite = (
    all_results
    .groupby(["metric", "mechanism"])
    .agg(
        mean_MAE        =("MAE",          "mean"),
        mean_OVERLAP    =("OVERLAP_at_k", "mean"),
        mean_TREND_fn   =("TREND_fn_rate","mean"),
    )
    .reset_index()
)

for metric in METRICS:
    sub = composite[composite["metric"] == metric].copy()
    sub["score"] = (
          0.50 * safe_norm(sub["mean_MAE"])
        + 0.25 * (1 - safe_norm(sub["mean_OVERLAP"].fillna(0.5)))
        + 0.25 * safe_norm(sub["mean_TREND_fn"].fillna(0.5))
    )
    sub = sub.sort_values("score")
    print(f"\n{metric} composite ranking (lower score = better):")
    print(sub[["mechanism","mean_MAE","mean_OVERLAP","mean_TREND_fn","score"]].to_string(index=False))


ALERT_COUNT composite ranking (lower score = better):
mechanism  mean_MAE  mean_OVERLAP  mean_TREND_fn    score
   smooth  1.622629           1.0            0.0 0.250000
     tree  2.023461           1.0            0.0 0.626631
    naive  2.154758           1.0            0.0 0.750000

TOTAL_FLOWS composite ranking (lower score = better):
mechanism  mean_MAE  mean_OVERLAP  mean_TREND_fn   score
   smooth  1.804629           1.0            0.0 0.25000
     tree  2.218734           1.0            0.0 0.58645
    naive  2.420034           1.0            0.0 0.75000

TOTAL_BYTES composite ranking (lower score = better):
mechanism     mean_MAE  mean_OVERLAP  mean_TREND_fn    score
   smooth 12189.125725           1.0            0.0 0.250000
     tree 15331.319960           1.0            0.0 0.635064
    naive 16269.224202           1.0            0.0 0.750000


## 9 — Clipping Sensitivity  *(multi-pct grid)*

Uses `results_*_multi.csv` (4 clip percentiles, 7 seeds) to show the
bias-variance tradeoff for `TOTAL_BYTES`.

Key finding from analysis: **70th-pct clip gives ~15–21× lower MAE** than the
default 90th-pct across all mechanisms, justifying pinning the hardened run to [70].

In [13]:
if multi_results.empty:
    print("Multi-pct CSVs not available — run v2 strategy notebooks to populate clipping analysis.")
else:
    clip_mae = (
        multi_results[multi_results["metric"] == "TOTAL_BYTES"]
        .groupby(["bytes_clip_pct", "mechanism"])["MAE"]
        .mean()
        .unstack("mechanism")
        .sort_index()
    )
    print("TOTAL_BYTES mean MAE by clip percentile:")
    print(clip_mae.round(2).to_string())

    # B_bytes_cap per percentile (noise sensitivity)
    b_map = (
        multi_results[multi_results["metric"] == "TOTAL_BYTES"]
        .groupby("bytes_clip_pct")["B_bytes_cap"]
        .first()
        .sort_index()
    )
    print("\nB_CLIP (noise sensitivity) per clip percentile:")
    for pct, b in b_map.items():
        print(f"  {int(pct)}th pct → B_CLIP = {b:,.0f} bytes")

    # Plot: clip pct vs MAE for each mechanism
    fig, ax = plt.subplots(figsize=(7, 4))
    for mech, color in MECH_COLORS.items():
        if mech in clip_mae.columns:
            ax.plot(clip_mae.index, clip_mae[mech], marker="o",
                    label=mech.capitalize(), color=color, lw=2)
    ax.set_xlabel("Bytes clip percentile")
    ax.set_ylabel("Mean MAE — TOTAL_BYTES")
    ax.set_title("Clipping Sensitivity: TOTAL_BYTES MAE vs Clip Percentile")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "clipping_sensitivity.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nSaved → {PLOTS_DIR / 'clipping_sensitivity.png'}")

    # Improvement ratio: pct_90 vs pct_70
    if 70 in clip_mae.index and 90 in clip_mae.index:
        ratio = clip_mae.loc[90] / clip_mae.loc[70]
        print("\nMAE ratio (90th pct / 70th pct) — how much worse is the default clip?")
        print(ratio.round(2).to_string())

TOTAL_BYTES mean MAE by clip percentile:
mechanism          naive    smooth      tree
bytes_clip_pct                              
70               1721.62   1427.50   1695.72
80              10379.93   7729.15   9787.41
90              24980.93  18243.95  22710.92
95              27994.41  21355.90  27131.23

B_CLIP (noise sensitivity) per clip percentile:
  70th pct → B_CLIP = 422 bytes
  80th pct → B_CLIP = 1,511 bytes
  90th pct → B_CLIP = 5,831 bytes
  95th pct → B_CLIP = 9,952 bytes

Saved → C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned\eval_plots\clipping_sensitivity.png

MAE ratio (90th pct / 70th pct) — how much worse is the default clip?
mechanism
naive     14.51
smooth    12.78
tree      13.39


C:\Users\Siddhartha\AppData\Local\Temp\ipykernel_25448\2619439352.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
